## The State of Tax Justice: Estimate misalignment for 2018

- Author: Mario Cuenda García, based on Alison Schultz, based on Javier Garcia Bernado's work
- Created: 4 November 2024
- Last updated:

**Description**
- This notebook is the third out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
    - Details on the misalignment method and its background can be found here: https://www.sciencedirect.com/science/article/pii/S0305750X23003455. 
    - The working paper version is here: https://www.econstor.eu/bitstream/10419/286362/1/wp-2023-33.pdf 

- This notebook estimates profit misalignment based on different formulas. It uses the dataset **"data/final/cbcr_main.csv"** (for the estimation with imputed values) or the dataset **"data/final/cbcr_main_noimputation_allsubgroupsonly.csv"** (for the estimation without imputed values). 

**Outline**
1. Define misalignment
2. Calculate misalignment for sample with full information.
3. Calculate misalignment for samples with imputed data and aggregate results. 

**To dos before running this notebook**
- Run the notebooks 1_clean and 2_impute_missings. Note the requirements and instructions given in these notebooks.

**To dos in this notebook**

Change the formula to any required formula in each section and adapt the output name of the csvs. The formulas I have used are like follows, where sales refer to unrelated party revenues and assets to tangible assets excluding cash.

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]

- Adjust the input path in 5.1 to the bootstrapped sample you use

## 0. Load packages

In [37]:
# Packages
import pandas as pd
import numpy as np
import tjn_tools
from config import *
import os

# Show columns and select data format
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.0f}'.format

## Step 1. Generate the template datasets

### Step 1.1. Generate the dataset with Unique ISO parents

In [38]:
# Open the original dataset
iso_parents = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep just the following columns: iso_parent and year
iso_parents = iso_parents[['iso_parent', 'year']]

# Keep every unique combination of iso_parent and year
iso_parents = iso_parents.drop_duplicates(subset=['iso_parent', 'year'])

# Sort by year, then iso_parent
iso_parents = iso_parents.sort_values(by=['year', 'iso_parent'])

# Filter by year
iso_parents_2016= iso_parents[iso_parents['year'] == 2016]
iso_parents_2017= iso_parents[iso_parents['year'] == 2017]
iso_parents_2018= iso_parents[iso_parents['year'] == 2018]
iso_parents_2019= iso_parents[iso_parents['year'] == 2019]
iso_parents_2020= iso_parents[iso_parents['year'] == 2020]
iso_parents_2021= iso_parents[iso_parents['year'] == 2021]

# OPTIONAL: Print the count of how many unique iso_partner values there are
print(iso_parents_2016['iso_parent'].nunique())
print(iso_parents_2017['iso_parent'].nunique())
print(iso_parents_2018['iso_parent'].nunique())
print(iso_parents_2019['iso_parent'].nunique())
print(iso_parents_2020['iso_parent'].nunique())
print(iso_parents_2021['iso_parent'].nunique())

iso_parents_2018

26
38
46
50
52
52


,iso_parent,year
152,ARG,2018
266,AUS,2018
762,AUT,2018
836,BEL,2018
1065,BMU,2018
1646,BRA,2018
1895,CAN,2018
1983,CHE,2018
2724,CHL,2018
2818,CHN,2018


### Step 1.2. Generate the dataset with unique iso_partners

In [39]:
## Download the relevant dataset
iso_partners = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
iso_partners = iso_partners[~iso_partners['iso_partner'].isin(non_countries)]

# Keep just the following columns: iso_partner and year
iso_partners = iso_partners[['iso_partner', 'year']]

# Sort by year, then iso_partner
iso_partners = iso_partners.sort_values(by=['year', 'iso_partner'])

# Keep every unique combination of iso_partner and year
iso_partners = iso_partners.drop_duplicates(subset=['iso_partner', 'year'])

# Filter by year
iso_partners_2016= iso_partners[iso_partners['year'] == 2016]
iso_partners_2017= iso_partners[iso_partners['year'] == 2017]
iso_partners_2018= iso_partners[iso_partners['year'] == 2018]
iso_partners_2019= iso_partners[iso_partners['year'] == 2019]
iso_partners_2020= iso_partners[iso_partners['year'] == 2020]
iso_partners_2021= iso_partners[iso_partners['year'] == 2021]


# Optional: Print the count of how many unique iso_partner values there are 
print(iso_partners_2016['iso_partner'].nunique())
print(iso_partners_2017['iso_partner'].nunique())
print(iso_partners_2018['iso_partner'].nunique())
print(iso_partners_2019['iso_partner'].nunique())
print(iso_partners_2020['iso_partner'].nunique())
print(iso_partners_2021['iso_partner'].nunique())

183
215
213
210
212
211


### Step 1.3. Generate the template dataset

In [40]:
# Perform a cross join to merge all values of iso_partners_ with each value of iso_parents
iso_combinations_2016 = iso_parents_2016.assign(key=1).merge(iso_partners_2016.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2017 = iso_parents_2017.assign(key=1).merge(iso_partners_2017.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2018 = iso_parents_2018.assign(key=1).merge(iso_partners_2018.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2019 = iso_parents_2019.assign(key=1).merge(iso_partners_2019.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2020 = iso_parents_2020.assign(key=1).merge(iso_partners_2020.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2021 = iso_parents_2021.assign(key=1).merge(iso_partners_2021.assign(key=1), on='key').drop('key', axis=1)

# Concatenate all the years
template_dataset = pd.concat([iso_combinations_2016, iso_combinations_2017, iso_combinations_2018, iso_combinations_2019, iso_combinations_2020, iso_combinations_2021])

# Drop year_y
template_dataset = template_dataset.drop(columns=['year_y'])
# Rename year_x to year
template_dataset = template_dataset.rename(columns={'year_x': 'year'})
# Order columns by iso_parent then iso_partner then year
template_dataset = template_dataset[['iso_parent', 'iso_partner', 'year']]
# Generate new column called cbcr_estimates
template_dataset['cbcr_estimates'] = np.nan

template_dataset

,iso_parent,iso_partner,year,cbcr_estimates
0,AUS,ABW,2016,NaN
1,AUS,AFG,2016,NaN
2,AUS,AGO,2016,NaN
3,AUS,ALB,2016,NaN
4,AUS,AND,2016,NaN
...,...,...,...,...
10967,ZAF,XKV,2021,NaN
10968,ZAF,YEM,2021,NaN
10969,ZAF,ZAF,2021,NaN
10970,ZAF,ZMB,2021,NaN


## Step 2. Define the misalignment formula

In [41]:
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                         'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5, 0, 0, .5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=0.15): 

    # Create variable with positive profits only for calculating shares
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var]
    cbcr_data.loc[cbcr_data[profit_var] < 0, 'profit_var_pos'] = 0
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data.groupby('iso_parent')['profit_var_pos'].transform('sum')

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f"share_{var}")
            actual_weights.append(weights[i])
            cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # Set economic activity measure to zero if negative
            cbcr_data[f"share_{var}"] = cbcr_data[var] / cbcr_data.groupby('iso_parent')[var].transform('sum')

    # Calculate the share of economic activity
    cbcr_data["share_economy_partner_of_parent"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(1, min_count=len(actual_weights))
    # Set economic activity to 1% for those jurisdictions without economic activity but with reported profits
    cbcr_data.loc[(cbcr_data["share_economy_partner_of_parent"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_partner_of_parent"] = 0.01

    # Normalize the economic activity shares to sum to 1
    cbcr_data["share_economy_partner_of_parent"] = cbcr_data["share_economy_partner_of_parent"] / cbcr_data.groupby('iso_parent')["share_economy_partner_of_parent"].transform('sum')

    # Calculate theoretical profit and misaligned profit
    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_partner_of_parent"] * cbcr_data.groupby('iso_parent')[profit_var].transform('sum')
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]

    # Set positive misaligned profits to 0 if ETR exceeds the threshold (etr_max)
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0) & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0

    # Adjust misalignment per 'iso_parent'
    def adjust_misalignment(group):
        total_negative_misalignment = group.loc[group["misaligned_profit"] < 0, "misaligned_profit"].sum()
        total_positive_misalignment = group.loc[group["misaligned_profit"] > 0, "misaligned_profit"].sum()
        
        # Adjust negative misalignments to balance positive misalignments within each 'iso_parent'
        if total_negative_misalignment != 0:
            factor = - total_positive_misalignment / total_negative_misalignment
            group.loc[group["misaligned_profit"] < 0, "misaligned_profit"] *= factor
        
        return group

    # Apply the adjustment by grouping by 'iso_parent'
    cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)

    return cbcr_data

## Step 3. Calculate misalignment for sample with full information

### Step 3.1 Import data
- Import data without imputed values. This is only the data from the sample of reporting countries that actually is reported on a country basis, i.e. excluding aggregated country groups and data from reporting countries that do not report on a country by country basis, but just by continents.

In [42]:
cbcr_sample = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
cbcr_sample = cbcr_sample[~cbcr_sample['iso_partner'].isin(non_countries)]

### Step 3.2 Exclude countries that do not report truly country-by-country


- In the 2024 data, the following reporting countries do not report country-by-country. We exclude those from the "clean" analysis where we only use values that are actually in the data.
    - Austria: Only continents in all years
    - Czechia: Only Czechia versus rest of the world from 2019 to 2021
    - Finland: Only Finland and rest of the world between 2016 and 2018 and Finland and continents between 2019 and 2021
    - Greece: Only Greece and continents between 2017 and 2019
    - Hungary: Only Hungary versus rest of the world between 2018 and 2021
    - Isle of Man: Only continents between 2017 and 2020
    - Ireland: Only Ireland versus rest of the world in all years
    - Korea: Only Korea and rest of the world betweem 2016 and 2018 and Korea and continents between 2019 and 2021
    - Macau: Only Macau versus rest of the world between 2019 and 2021
    - Mauritius: Only Mauritius and continents between 2019 and 2021
    - Morocco: Only Morocco and continents in 2021
    - Netherlands: Only Netherlands versus rest of the world between 2016 and 2017
    - Norway: Only Norway and continents 2016 and 2017
    - New Zealand: Only New Zealand versus rest of the world between 2018 and 2021
    - Poland: Only Poland and continents 2019 to 2021
    - Sweden: Only Sweden and continents in all years
    - United Kingdom: Only UK and continents between 2017 and 2021

In [43]:
# Define the conditions for exclusion
exclusion_conditions = [
    ('AUT', 2016, 2021),                # Austria: all years
    ('CZE', 2019, 2021),                # Czechia: from 2019 to 2021
    ('FIN', 2016, 2021),                # Finland: all years
    ('GRC', 2017, 2019),                # Greece: between 2017 and 2019
    ('HUN', 2018, 2021),                # Hungary: between 2018 and 2021
    ('IMN', 2017, 2020),                # Isle of Man: between 2017 and 2020
    ('IRL', 2016, 2021),                # Ireland: all years
    ('KOR', 2016, 2021),                # Korea: all years
    ('MAC', 2019, 2021),                # Macau: between 2019 and 2021
    ('MUS', 2019, 2021),                # Mauritius: between 2019 and 2021
    ('MAR', 2021, 2021),                # Morocco: 2021
    ('NLD', 2016, 2017),                # Netherlands: between 2016 and 2017
    ('NOR', 2016, 2017),                # Norway: 2016 and 2017
    ('NZL', 2018, 2021),                # New Zealand: between 2018 and 2021
    ('POL', 2019, 2021),                # Poland: 2019 to 2021
    ('SWE', 2016, 2021),                # Sweden: all years
    ('GBR', 2017, 2021)                 # United Kingdom: between 2017 and 2021
]

# Iterate through the exclusion conditions
for iso_parent, start_year, end_year in exclusion_conditions:
    cbcr_sample = cbcr_sample[~((cbcr_sample['iso_parent'] == iso_parent) & 
                                     (cbcr_sample['year'].between(start_year, end_year)))]

### Step 3.3 Calculate misalignment for sample countries with full information

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]


The end of this cell shows, among other things, the misaligned profits and the theoretical profits, as well as the CBCR variables.

**More importantly, if for whatever reason we wanted to just run this cell without the "bad reporters", we could just modify the cell to run like the cell from Step 5.2. in order to obtain a dataset of profit shifting without the "bad reporters". In a way, this cell is not really necessary for the rest of the notebook. It just shows an intermediary steps if we want to calculate the misalignment for the "good reporters" only.**

In [44]:
misalignment_2018 = cbcr_sample[cbcr_sample['year'] == 2018].copy()
misalignment_2018 = calculate_misalignment(misalignment_2018, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Keep only the first occurrence of these unique variables for each 'iso_partner'
unique_columns = misalignment_2018.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

# Keep iso_parent, iso_partner, year, misaligned_profit, theoretical_profit, profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
misalignment_2018 = misalignment_2018[['iso_parent', 'iso_partner', 'year', 'misaligned_profit', 'theoretical_profit', 'profit_loss_before_income_tax_corrected', 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']]

# Show if iso_partner = USA
misalignment_2018[misalignment_2018['iso_partner'] == 'USA']

C:\Users\aliso\AppData\Local\Temp\ipykernel_10016\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip
16,ARG,USA,2018,"-50,270,836","43,011,110","-122,088,143",189,"494,648,758","2,461,955,134","9,272,650","3,268,852,476","534,347,546","39,698,788",0
90,AUS,USA,2018,0,"4,568,274,907","6,676,824,919","81,852","45,781,753,175","39,993,863,103","4,015,793,357","165,868,000,000","57,927,263,558","12,145,510,383",19
111,BEL,USA,2018,"-15,741,957,618","17,491,257,618","1,749,300,000","51,100","31,823,400,000","13,855,500,000","2,507,049,804","63,875,800,000","40,451,600,000","8,628,100,000",11
203,BMU,USA,2018,"-1,987,661,678","4,663,699,762","804,038,706","94,844","62,022,607,857","30,278,731,390","4,653,202,184","41,642,634,608","75,440,646,403","13,418,038,543",NaN
241,BRA,USA,2018,"-620,726,507","2,396,369,802","1,649,718,637","27,775","17,694,325,644","13,208,335,520","1,362,687,051","20,728,681,448","37,254,157,031","19,559,831,387",4
254,CAN,USA,2018,"-22,025,190,304","52,323,397,821","30,164,018,000","654,120","352,631,000,000","321,723,000,000","32,092,199,957","776,666,000,000","445,171,000,000","92,539,590,000",NaN
410,CHE,USA,2018,0,"16,391,862,634","19,614,658,236","303,601","216,993,000,000","58,812,996,390","14,895,162,966","349,535,000,000","254,592,000,000","37,598,967,733",80
538,CHN,USA,2018,"-3,951,363,035","13,956,066,693","1,066,105,792","110,677","88,161,221,710","46,762,421,195","5,429,995,130","55,595,963,254","101,622,000,000","12,623,165,302",52
685,CYM,USA,2018,"-1,126,995,092","4,681,142,649","278,609,840","56,028","48,161,756,445","9,459,255,428","2,748,825,566","20,466,570,658","50,557,510,578","2,417,815,354",35
844,DEU,USA,2018,0,"36,442,465,321","41,081,128,000","672,958","454,404,000,000","253,864,000,000","33,016,423,131","440,025,000,000","584,184,000,000","129,780,000,000",202


## Step 4. Generate the dataset for the "bad reporters"

For the "bad" reporters, we are going to assume that their MNEs behave like the "average" MNE in the countries that report correctly. To do that we need to:

1. First aggregate the variables reported by the CBCR by partner countries. For instance, we see that on aggregate, there are 80m employees reported.
2. Then we look at the share corresponding by partners. For instance, 18m employees are reported in the USA. how many does the USA have. In short, roughly 24% of all employees reported are in the USA.
3. We then assume that 24% of the toal employees reported by the "bad" reporters are assigned to the US. And we repeat with all of them



### Step 4.1. Generate the Total Sums of Variables, and the Total Sums by Partners.

- The first bloc of lines calculates the total of the variables, and generates a new variable (e.g. total_n_employees) in the dataset.
- The second bloc of lines groups by iso_partner and calculates the total sums by partners. (e.g. how many employees are reported by the CBCR countries, say, in the USA)
- The third bloc of lines merges the total sums by partners to the dataset

In [45]:
# Calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_before_income_tax_corrected = misalignment_2018['profit_loss_before_income_tax_corrected'].sum()
total_n_employees = misalignment_2018['n_employees'].sum()
total_unrelated_party_revenues = misalignment_2018['unrelated_party_revenues'].sum()
total_tangible_assets_except_cash = misalignment_2018['tangible_assets_except_cash'].sum()
total_payroll = misalignment_2018['payroll'].sum()
total_stated_capital = misalignment_2018['stated_capital'].sum()
total_total_revenues = misalignment_2018['total_revenues'].sum()
total_related_party_revenues = misalignment_2018['related_party_revenues'].sum()
total_holding_or_managing_ip = misalignment_2018['holding_or_managing_ip'].sum()

misalignment_2018['total_profit_loss_before_income_tax_corrected'] = total_profit_loss_before_income_tax_corrected
misalignment_2018['total_n_employees'] = total_n_employees
misalignment_2018['total_unrelated_party_revenues'] = total_unrelated_party_revenues
misalignment_2018['total_tangible_assets_except_cash'] = total_tangible_assets_except_cash
misalignment_2018['total_payroll'] = total_payroll
misalignment_2018['total_stated_capital'] = total_stated_capital
misalignment_2018['total_total_revenues'] = total_total_revenues
misalignment_2018['total_related_party_revenues'] = total_related_party_revenues
misalignment_2018['total_holding_or_managing_ip'] = total_holding_or_managing_ip

# Group by iso_partner and calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_by_partner = misalignment_2018.groupby('iso_partner')['profit_loss_before_income_tax_corrected'].sum().reset_index()
total_profit_loss_by_partner = total_profit_loss_by_partner.rename(columns={'profit_loss_before_income_tax_corrected': 'total_profit_loss_by_partner'})

total_n_employees_by_partner = misalignment_2018.groupby('iso_partner')['n_employees'].sum().reset_index()
total_n_employees_by_partner = total_n_employees_by_partner.rename(columns={'n_employees': 'total_n_employees_by_partner'})

total_unrelated_party_revenues_by_partner = misalignment_2018.groupby('iso_partner')['unrelated_party_revenues'].sum().reset_index()
total_unrelated_party_revenues_by_partner = total_unrelated_party_revenues_by_partner.rename(columns={'unrelated_party_revenues': 'total_unrelated_party_revenues_by_partner'})

total_tangible_assets_except_cash_by_partner = misalignment_2018.groupby('iso_partner')['tangible_assets_except_cash'].sum().reset_index()
total_tangible_assets_except_cash_by_partner = total_tangible_assets_except_cash_by_partner.rename(columns={'tangible_assets_except_cash': 'total_tangible_assets_except_cash_by_partner'})

total_payroll_by_partner = misalignment_2018.groupby('iso_partner')['payroll'].sum().reset_index()
total_payroll_by_partner = total_payroll_by_partner.rename(columns={'payroll': 'total_payroll_by_partner'})

total_stated_capital_by_partner = misalignment_2018.groupby('iso_partner')['stated_capital'].sum().reset_index()
total_stated_capital_by_partner = total_stated_capital_by_partner.rename(columns={'stated_capital': 'total_stated_capital_by_partner'})

total_total_revenues_by_partner = misalignment_2018.groupby('iso_partner')['total_revenues'].sum().reset_index()
total_total_revenues_by_partner = total_total_revenues_by_partner.rename(columns={'total_revenues': 'total_total_revenues_by_partner'})

total_related_party_revenues_by_partner = misalignment_2018.groupby('iso_partner')['related_party_revenues'].sum().reset_index()
total_related_party_revenues_by_partner = total_related_party_revenues_by_partner.rename(columns={'related_party_revenues': 'total_related_party_revenues_by_partner'})

total_holding_or_managing_ip_by_partner = misalignment_2018.groupby('iso_partner')['holding_or_managing_ip'].sum().reset_index()
total_holding_or_managing_ip_by_partner = total_holding_or_managing_ip_by_partner.rename(columns={'holding_or_managing_ip': 'total_holding_or_managing_ip_by_partner'})

# Merge the total profit loss by partner back into the misalignment_2018 dataframe
misalignment_2018 = misalignment_2018.merge(total_profit_loss_by_partner, on='iso_partner', how='left')
misalignment_2018 = misalignment_2018.merge(total_n_employees_by_partner, on='iso_partner', how='left')
misalignment_2018 = misalignment_2018.merge(total_unrelated_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2018 = misalignment_2018.merge(total_tangible_assets_except_cash_by_partner, on='iso_partner', how='left')
misalignment_2018 = misalignment_2018.merge(total_payroll_by_partner, on='iso_partner', how='left')
misalignment_2018 = misalignment_2018.merge(total_stated_capital_by_partner, on='iso_partner', how='left')
misalignment_2018 = misalignment_2018.merge(total_total_revenues_by_partner, on='iso_partner', how='left')
misalignment_2018 = misalignment_2018.merge(total_related_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2018 = misalignment_2018.merge(total_holding_or_managing_ip_by_partner, on='iso_partner', how='left')

misalignment_2018[misalignment_2018['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner
16,ARG,USA,2018,"-50,270,836","43,011,110","-122,088,143",189,"494,648,758","2,461,955,134","9,272,650","3,268,852,476","534,347,546","39,698,788",0,"5,063,961,381,539","132,892,417","48,400,824,491,538","36,736,934,833,716","3,487,464,054,782","58,738,472,108,198","68,081,106,384,255","19,606,343,011,562","16,114","574,894,436,078","28,160,040","14,178,824,603,989","7,384,912,551,289","1,381,577,765,177","17,795,046,846,961","18,417,562,320,492","4,238,036,397,261","1,285"
90,AUS,USA,2018,0,"4,568,274,907","6,676,824,919","81,852","45,781,753,175","39,993,863,103","4,015,793,357","165,868,000,000","57,927,263,558","12,145,510,383",19,"5,063,961,381,539","132,892,417","48,400,824,491,538","36,736,934,833,716","3,487,464,054,782","58,738,472,108,198","68,081,106,384,255","19,606,343,011,562","16,114","574,894,436,078","28,160,040","14,178,824,603,989","7,384,912,551,289","1,381,577,765,177","17,795,046,846,961","18,417,562,320,492","4,238,036,397,261","1,285"
111,BEL,USA,2018,"-15,741,957,618","17,491,257,618","1,749,300,000","51,100","31,823,400,000","13,855,500,000","2,507,049,804","63,875,800,000","40,451,600,000","8,628,100,000",11,"5,063,961,381,539","132,892,417","48,400,824,491,538","36,736,934,833,716","3,487,464,054,782","58,738,472,108,198","68,081,106,384,255","19,606,343,011,562","16,114","574,894,436,078","28,160,040","14,178,824,603,989","7,384,912,551,289","1,381,577,765,177","17,795,046,846,961","18,417,562,320,492","4,238,036,397,261","1,285"
203,BMU,USA,2018,"-1,987,661,678","4,663,699,762","804,038,706","94,844","62,022,607,857","30,278,731,390","4,653,202,184","41,642,634,608","75,440,646,403","13,418,038,543",NaN,"5,063,961,381,539","132,892,417","48,400,824,491,538","36,736,934,833,716","3,487,464,054,782","58,738,472,108,198","68,081,106,384,255","19,606,343,011,562","16,114","574,894,436,078","28,160,040","14,178,824,603,989","7,384,912,551,289","1,381,577,765,177","17,795,046,846,961","18,417,562,320,492","4,238,036,397,261","1,285"
241,BRA,USA,2018,"-620,726,507","2,396,369,802","1,649,718,637","27,775","17,694,325,644","13,208,335,520","1,362,687,051","20,728,681,448","37,254,157,031","19,559,831,387",4,"5,063,961,381,539","132,892,417","48,400,824,491,538","36,736,934,833,716","3,487,464,054,782","58,738,472,108,198","68,081,106,384,255","19,606,343,011,562","16,114","574,894,436,078","28,160,040","14,178,824,603,989","7,384,912,551,289","1,381,577,765,177","17,795,046,846,961","18,417,562,320,492","4,238,036,397,261","1,285"
254,CAN,USA,2018,"-22,025,190,304","52,323,397,821","30,164,018,000","654,120","352,631,000,000","321,723,000,000","32,092,199,957","776,666,000,000","445,171,000,000","92,539,590,000",NaN,"5,063,961,381,539","132,892,417","48,400,824,491,538","36,736,934,833,716","3,487,464,054,782","58,738,472,108,198","68,081,106,384,255","19,606,343,011,562","16,114","574,894,436,078","28,160,040","14,178,824,603,989","7,384,912,551,289","1,381,577,765,177","17,795,046,846,961","18,417,562,320,492","4,238,036,397,261","1,285"
410,CHE,USA,2018,0,"16,391,862,634","19,614,658,236","303,601","216,993,000,000","58,812,996,390","14,895,162,966","349,535,000,000","254,592,000,000","37,598,967,733",80,"

### Step 4.2. Calculate the shares for all the variables

In [46]:
# Final Misalignment
final_misalignment_2018 = misalignment_2018

# Calculate the shares for all variables
final_misalignment_2018['share_reported_total_profit_loss_by_partner'] = misalignment_2018['total_profit_loss_by_partner'] / misalignment_2018['total_profit_loss_before_income_tax_corrected']
final_misalignment_2018['share_reported_total_n_employees_by_partner'] = misalignment_2018['total_n_employees_by_partner'] / misalignment_2018['total_n_employees']
final_misalignment_2018['share_reported_total_unrelated_party_revenues_by_partner'] = misalignment_2018['total_unrelated_party_revenues_by_partner'] / misalignment_2018['total_unrelated_party_revenues']
final_misalignment_2018['share_reported_total_tangible_assets_except_cash_by_partner'] = misalignment_2018['total_tangible_assets_except_cash_by_partner'] / misalignment_2018['total_tangible_assets_except_cash']
final_misalignment_2018['share_reported_total_payroll_by_partner'] = misalignment_2018['total_payroll_by_partner'] / misalignment_2018['total_payroll']
final_misalignment_2018['share_reported_total_stated_capital_by_partner'] = misalignment_2018['total_stated_capital_by_partner'] / misalignment_2018['total_stated_capital']
final_misalignment_2018['share_reported_total_total_revenues_by_partner'] = misalignment_2018['total_total_revenues_by_partner'] / misalignment_2018['total_total_revenues']
final_misalignment_2018['share_reported_total_related_party_revenues_by_partner'] = misalignment_2018['total_related_party_revenues_by_partner'] / misalignment_2018['total_related_party_revenues']
final_misalignment_2018['share_reported_total_holding_or_managing_ip_by_partner'] = misalignment_2018['total_holding_or_managing_ip_by_partner'] / misalignment_2018['total_holding_or_managing_ip']

# Give me column 'share reported' with 2 decimals
pd.options.display.float_format = '{:,.2f}'.format

final_misalignment_2018[final_misalignment_2018['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
16,ARG,USA,2018,"-50,270,836.25","43,011,110.22","-122,088,143.20",189.00,"494,648,758.50","2,461,955,134.00","9,272,649.96","3,268,852,476.00","534,347,546.20","39,698,787.66",0.00,"5,063,961,381,539.22","132,892,417.10","48,400,824,491,537.50","36,736,934,833,715.97","3,487,464,054,781.80","58,738,472,108,198.50","68,081,106,384,255.22","19,606,343,011,562.18","16,114.00","574,894,436,077.94","28,160,040.41","14,178,824,603,988.89","7,384,912,551,289.05","1,381,577,765,177.12","17,795,046,846,961.30","18,417,562,320,492.21","4,238,036,397,261.08","1,285.00",0.11,0.21,0.29,0.20,0.40,0.30,0.27,0.22,0.08
90,AUS,USA,2018,0.00,"4,568,274,907.42","6,676,824,919.00","81,852.00","45,781,753,175.00","39,993,863,103.00","4,015,793,357.28","165,868,000,000.00","57,927,263,558.00","12,145,510,383.00",19.00,"5,063,961,381,539.22","132,892,417.10","48,400,824,491,537.50","36,736,934,833,715.97","3,487,464,054,781.80","58,738,472,108,198.50","68,081,106,384,255.22","19,606,343,011,562.18","16,114.00","574,894,436,077.94","28,160,040.41","14,178,824,603,988.89","7,384,912,551,289.05","1,381,577,765,177.12","17,795,046,846,961.30","18,417,562,320,492.21","4,238,036,397,261.08","1,285.00",0.11,0.21,0.29,0.20,0.40,0.30,0.27,0.22,0.08
111,BEL,USA,2018,"-15,741,957,617.54","17,491,257,617.54","1,749,300,000.00","51,100.00","31,823,400,000.00","13,855,500,000.00","2,507,049,804.00","63,875,800,000.00","40,451,600,000.00","8,628,100,000.00",11.00,"5,063,961,381,539.22","132,892,417.10","48,400,824,491,537.50","36,736,934,833,715.97","3,487,464,054,781.80","58,738,472,108,198.50","68,081,106,384,255.22","19,606,343,011,562.18","16,114.00","574,894,436,077.94","28,160,040.41","14,178,824,603,988.89","7,384,912,551,289.05","1,381,577,765,177.12","17,795,046,846,961.30","18,417,562,320,492.21","4,238,036,397,261.08","1,285.00",0.11,0.21,0.29,0.20,0.40,0.30,0.27,0.22,0.08
203,BMU,USA,2018,"-1,987,661,677.86","4,663,699,761.58","804,038,705.90","94,844.00","62,022,607,857.00","30,278,731,390.00","4,653,202,184.16","41,642,634,608.00","75,440,646,403.00","13,418,038,543.00",NaN,"5,063,961,381,539.22","132,892,417.10","48,400,824,491,537.50","36,736,934,833,715.97","3,487,464,054,781.80","58,738,472,108,198.50","68,081,106,384,255.22","19,606,343,011,562.18","16,114.00","574,894,436,077.94","28,160,040.41","14,178,824,603,988.89","7,384,912,551,289.05","1,381,577,765,177.12","17,795,046,846,961.30","18,417,562,320,492.21","4,238,036,397,261.08","1,285.00",0.11,0.21,0.29,0.20,0.40,0.30,0.27,0.22,0.08
241,BRA,USA,2018,"-620,726,507.41","2,396,369,802.42","1,649,718,637.00","27,775.00","17,694,325,644.00","13,208,335,520.00","1,362,687,051.00","20,728,681,448.00","37,254,157,031.00","19,559,831,387.00",4.00,"5,063,961,381

### Step 4.3. Keep only the shares for the partners, and drop duplicates (effectively only keep one value for each iso_partner)

In [47]:
# Keep iso_partner and share_reported_total_profit_loss_by_partner	share_reported_total_n_employees_by_partner	share_reported_total_unrelated_party_revenues_by_partner	share_reported_total_tangible_assets_except_cash_by_partner	share_reported_total_payroll_by_partner	share_reported_total_stated_capital_by_partner	share_reported_total_total_revenues_by_partner	share_reported_total_related_party_revenues_by_partner	share_reported_total_holding_or_managing_ip_by_partner
shares_reported_2018 = final_misalignment_2018[['iso_partner', 'share_reported_total_profit_loss_by_partner', 'share_reported_total_n_employees_by_partner', 'share_reported_total_unrelated_party_revenues_by_partner', 'share_reported_total_tangible_assets_except_cash_by_partner', 'share_reported_total_payroll_by_partner', 'share_reported_total_stated_capital_by_partner', 'share_reported_total_total_revenues_by_partner', 'share_reported_total_related_party_revenues_by_partner', 'share_reported_total_holding_or_managing_ip_by_partner']]

# Drop duplicates
shares_reported_2018 = shares_reported_2018.drop_duplicates()

shares_reported_2018[shares_reported_2018['iso_partner'] == 'USA']

,iso_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
16,USA,0.11,0.21,0.29,0.20,0.40,0.30,0.27,0.22,0.08


### Step 4.4. Bring back the excluded countries

- The key here is the third command, where we sum by iso_parent. We basically assume that all countries report for the rest of the world, without caring about continents. **This could be improved and changed**. 
- Once that sum is done, we combine all potential iso_combinations for 2016 (created in Step 1), and we keep all combinations for the "excluded countries".
- Note in the test view, that no matter who is the iso_partner, the number in the variables will be the same because we have aggregated them. The next cells will now create the right shares.


In [48]:
excluded_2018 = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep if year == 2018 and iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_2018 = excluded_2018[excluded_2018['year'] == 2018]
excluded_2018 = excluded_2018[excluded_2018['iso_parent'].isin(['AUT', 'FIN', 'GRC', 'HUN', 'IMN', 'IRL', 'KOR', 'NZL', 'SWE', 'GBR'])]

# 3. Sum by iso_parent: 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected'
excluded_2018 = excluded_2018.groupby('iso_parent').agg({'n_employees': 'sum', 'unrelated_party_revenues': 'sum', 'tangible_assets_except_cash': 'sum', 'payroll': 'sum', 'stated_capital': 'sum', 'total_revenues': 'sum', 'related_party_revenues': 'sum', 'holding_or_managing_ip': 'sum', 'profit_loss_before_income_tax_corrected': 'sum'}).reset_index()

# Merge iso_combinations_2018 with excluded_2018. 
excluded_jurisdictions_2018 = pd.merge(iso_combinations_2018, excluded_2018, on='iso_parent', how='left')

# Drop year_y
excluded_jurisdictions_2018 = excluded_jurisdictions_2018.drop(columns=['year_y'])
# Rename year_x to year
excluded_jurisdictions_2018 = excluded_jurisdictions_2018.rename(columns={'year_x': 'year'})

# Keep if iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_jurisdictions_2018 = excluded_jurisdictions_2018[excluded_jurisdictions_2018['iso_parent'].isin(['AUT', 'FIN', 'GRC', 'HUN', 'IMN', 'IRL', 'KOR', 'NZL', 'SWE', 'GBR'])]

excluded_jurisdictions_2018[excluded_jurisdictions_2018['iso_partner'] == 'USA']


,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
624,AUT,2018,USA,"1,736,516.00","545,843,630,468.00","309,726,071,117.00","12,860,357,111.23","193,328,986,761.10","697,760,785,968.00","151,917,450,287.50",502.00,"44,046,424,001.38"
3393,FIN,2018,USA,"596,405.00","254,606,000,000.00","95,453,548,613.00","7,180,598,507.76","306,822,000,000.00","358,428,000,000.00","103,823,560,679.00",139.00,"25,937,827,335.82"
3819,GBR,2018,USA,"13,100,663.00","4,162,077,024,858.00","2,930,552,060,879.00","122,159,395,651.10","8,955,333,087,344.00","5,643,297,816,868.00","1,481,550,809,852.00","2,154.00","435,484,435,723.00"
4032,GRC,2018,USA,"242,534.00","77,431,708,394.00","49,391,649,990.00","1,658,735,842.80","118,083,119,007.00","91,829,552,482.00","14,397,844,086.00",18.00,"642,741,566.52"
4458,HUN,2018,USA,"90,024.00","39,505,924,533.00","14,740,649,659.00","586,181,035.63","11,999,356,559.00","52,752,157,963.00","13,246,233,430.00",0.00,"4,702,088,570.16"
4884,IMN,2018,USA,"135,726.00","23,744,089,019.87","14,688,466,661.23","37,845,742.99","91,770,555,400.93","30,836,536,384.79","7,092,447,372.32",44.00,"1,808,770,671.59"
5310,IRL,2018,USA,"1,709,425.00","428,792,970,136.00","174,354,558,936.00","5,348,998,678.97","3,361,370,000,000.00","743,985,000,000.00","309,503,858,000.00",436.00,"80,543,905,702.97"
5949,KOR,2018,USA,"3,743,544.00","1,986,242,000,000.00","1,410,766,000,000.00","68,299,523,479.74","626,174,000,000.00","3,005,250,000,000.00","888,384,000,000.00",701.00,"173,724,784,601.61"
7653,NZL,2018,USA,"145,371.00","64,538,693,673.00","51,841,761,543.00","3,537,333,110.01","60,609,560,743.00","94,356,901,005.00","29,813,222,892.00",259.00,"3,714,563,911.07"
9357,SWE,2018,USA,"3,326,212.00","938,153,867,799.00","448,171,250,434.00","19,681,563,370.56","334,365,269,828.00","1,514,794,837,985.00","576,640,695,068.00","1,045.00","155,007,926,128.36"


### Step 4.5. Merge with the shares reported, and multiply the number

In [49]:
# Merge with share_reported_2018
excluded_jurisdictions_share_reported_2018 = pd.merge(excluded_jurisdictions_2018, shares_reported_2018, on='iso_partner', how='left')

# Multiply 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected' by share_reported
excluded_jurisdictions_share_reported_2018['n_employees'] = excluded_jurisdictions_share_reported_2018['n_employees'] * excluded_jurisdictions_share_reported_2018['share_reported_total_n_employees_by_partner']
excluded_jurisdictions_share_reported_2018['unrelated_party_revenues'] = excluded_jurisdictions_share_reported_2018['unrelated_party_revenues'] * excluded_jurisdictions_share_reported_2018['share_reported_total_unrelated_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2018['tangible_assets_except_cash'] = excluded_jurisdictions_share_reported_2018['tangible_assets_except_cash'] * excluded_jurisdictions_share_reported_2018['share_reported_total_tangible_assets_except_cash_by_partner']
excluded_jurisdictions_share_reported_2018['payroll'] = excluded_jurisdictions_share_reported_2018['payroll'] * excluded_jurisdictions_share_reported_2018['share_reported_total_payroll_by_partner']
excluded_jurisdictions_share_reported_2018['stated_capital'] = excluded_jurisdictions_share_reported_2018['stated_capital'] * excluded_jurisdictions_share_reported_2018['share_reported_total_stated_capital_by_partner']
excluded_jurisdictions_share_reported_2018['total_revenues'] = excluded_jurisdictions_share_reported_2018['total_revenues'] * excluded_jurisdictions_share_reported_2018['share_reported_total_total_revenues_by_partner']
excluded_jurisdictions_share_reported_2018['related_party_revenues'] = excluded_jurisdictions_share_reported_2018['related_party_revenues'] * excluded_jurisdictions_share_reported_2018['share_reported_total_related_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2018['holding_or_managing_ip'] = excluded_jurisdictions_share_reported_2018['holding_or_managing_ip'] * excluded_jurisdictions_share_reported_2018['share_reported_total_holding_or_managing_ip_by_partner']
excluded_jurisdictions_share_reported_2018['profit_loss_before_income_tax_corrected'] = excluded_jurisdictions_share_reported_2018['profit_loss_before_income_tax_corrected'] * excluded_jurisdictions_share_reported_2018['share_reported_total_profit_loss_by_partner']

# Drop share_reported columns
excluded_jurisdictions_dataset_2018 = excluded_jurisdictions_share_reported_2018.drop(columns=[
    'share_reported_total_profit_loss_by_partner',
    'share_reported_total_n_employees_by_partner',
    'share_reported_total_unrelated_party_revenues_by_partner',
    'share_reported_total_tangible_assets_except_cash_by_partner',
    'share_reported_total_payroll_by_partner',
    'share_reported_total_stated_capital_by_partner',
    'share_reported_total_total_revenues_by_partner',
    'share_reported_total_related_party_revenues_by_partner',
    'share_reported_total_holding_or_managing_ip_by_partner'
])

excluded_jurisdictions_dataset_2018[excluded_jurisdictions_dataset_2018['iso_partner'] == 'USA']

,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
198,AUT,2018,USA,"367,969.53","159,902,670,644.87","62,261,589,335.27","5,094,700,090.96","58,569,762,760.46","188,760,927,118.74","32,837,928,180.58",40.03,"5,000,441,784.54"
411,FIN,2018,USA,"126,378.84","74,585,791,772.09","19,188,212,419.13","2,844,632,972.03","92,952,909,187.37","96,963,318,870.74","22,442,126,448.13",11.08,"2,944,633,952.71"
624,GBR,2018,USA,"2,776,044.02","1,219,263,529,985.35","589,103,875,828.23","48,394,105,914.19","2,713,052,725,067.16","1,526,646,594,851.77","320,246,680,007.67",171.77,"49,439,077,479.62"
837,GRC,2018,USA,"51,393.21","22,683,303,923.83","9,928,781,962.65","657,117,183.92","35,773,736,686.48","24,842,083,149.39","3,112,186,053.39",1.44,"72,968,279.69"
1050,HUN,2018,USA,"19,076.18","11,573,099,852.25","2,963,187,026.18","232,218,790.64","3,635,251,385.28","14,270,716,332.67","2,863,258,047.15",0.00,"533,812,237.74"
1263,IMN,2018,USA,"28,760.48","6,955,734,269.63","2,952,697,123.39","14,992,795.97","27,802,243,979.43","8,342,018,230.57","1,533,077,845.87",3.51,"205,343,626.64"
1476,IRL,2018,USA,"362,228.92","125,613,155,950.37","35,049,009,300.56","2,119,034,784.23","1,018,340,015,889.20","201,265,678,992.85","66,901,237,753.68",34.77,"9,143,877,640.42"
1689,KOR,2018,USA,"793,260.84","581,861,512,379.82","283,594,251,602.35","27,057,225,975.54","189,701,830,238.68","812,991,769,717.47","192,029,881,580.88",55.90,"19,722,388,176.26"
1902,NZL,2018,USA,"30,804.27","18,906,347,719.76","10,421,306,982.54","1,401,333,661.38","18,361,900,371.00","25,525,791,176.48","6,444,318,742.23",20.65,"421,701,897.79"
2115,SWE,2018,USA,"704,827.76","274,828,358,459.09","90,092,042,448.23","7,796,957,878.15","101,297,249,094.70","409,788,116,160.82","124,644,573,054.70",83.33,"17,597,526,435.34"


## Step 5. Calculate Misalignment with all countries reporting in the CBCR

### Step 5.1. Concatenate the two samples

- Concatenate the cbcr_sample (without the "bad reporters") and the sample with the bad reporters.
- Ensure all required columns are included: 'iso_parent', 'year', 'iso_partner', 'n_employees', 'unrelated_party_revenues', 'tangible_assets_except_cash', 'payroll', 'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip', 'profit_loss_before_income_tax_corrected'.

In [50]:
final_misalignment_2018 = cbcr_sample[cbcr_sample['year'] == 2018].copy()

# Concatenate excluded_jurisdictions_dataset_2016
final_misalignment_2018 = pd.concat([final_misalignment_2018, excluded_jurisdictions_dataset_2018])

# Save the final misalignment dataset
#final_misalignment_2018.to_csv(f'{output_tables}/Datasets_to_Perform_Analysis/misalignment_dataset_2018.csv', index=False)

final_misalignment_2018[final_misalignment_2018['iso_partner'] == 'ZAF']

,iso_parent,parent_jurisdiction,iso_partner,partner_jurisdiction,year,unrelated_party_revenues,profit_loss_before_income_tax,adjusted_profit_loss_before_income_tax,income_tax_paid_on_cash_basis,income_tax_accrued_current_year,n_employees,tangible_assets_except_cash,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,n_cbcr,n_cbcr_groups,n_entities,profit_loss_before_income_tax_corrected,ln_profit_loss_before_income_tax_corrected,ln_unrelated_party_revenues,ln_n_employees,ln_tangible_assets_except_cash,ln_stated_capital,ln_total_revenues,ln_related_party_revenues,ln_holding_or_managing_ip,etr_domestic,etr_domestic_corrected,etr_foreign,etr_foreign_corrected,etr_average,etr_average_corrected,cit,gdp_current_usd,population,gdp,wage_monthly,payroll,ln_wage_monthly,ln_gdp_current_usd,ln_population,gvt_health_expenditure,ln_gvt_health_expenditure,tax_revenue_pct_gdp,tax_revenue_current_usd,cthi_2021_share,cthi_2021_score,region_tjn,ukt,gbr_oct,nld_oct,oecd_oct,oecd,eu
261,ARG,Argentina,ZAF,South Africa,2018,"1,704.05","-75,079.76",NaN,597.94,"7,982.22",1.00,0.00,"-80,948.86","1,704.05",0.00,0.00,2.00,2.00,2.00,"-75,079.76",0.00,7.44,0.69,0.00,0.00,7.44,0.00,0.00,0.11,0.15,0.20,0.22,0.12,0.17,0.28,"405,260,723,892.52","57,339,635.00",NaN,264.47,"3,173.66",5.58,26.73,17.86,"18,683,469,011.80",23.65,24.89,"100,889,227,920.36",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
748,AUS,Australia,ZAF,South Africa,2018,"1,650,501,387.00","444,999,541.00",NaN,"113,682,190.00","151,201,792.00","14,807.00","2,245,513,146.00","4,436,718,443.00","4,114,964,011.00","2,464,462,626.00",5.00,23.00,23.00,131.00,"444,999,541.00",19.91,21.22,9.60,21.53,22.21,22.14,21.63,1.79,0.11,0.15,0.20,0.22,0.12,0.17,0.28,"405,260,723,892.52","57,339,635.00",NaN,264.47,"46,992,442.85",5.58,26.73,17.86,"18,683,469,011.80",23.65,24.89,"100,889,227,920.36",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
1640,BMU,Bermuda,ZAF,South Africa,2018,"568,286,575.60","206,527,692.90",NaN,"62,010,201.57","61,224,843.71","2,561.00","82,051,958.40","407,188,674.50","846,721,063.90","278,434,489.30",NaN,21.00,21.00,NaN,"206,527,692.90",19.15,20.16,7.85,18.22,19.82,20.56,19.44,NaN,0.11,0.15,0.20,0.22,0.12,0.17,0.28,"405,260,723,892.52","57,339,635.00",NaN,264.47,"8,127,753.50",5.58,26.73,17.86,"18,683,469,011.80",23.65,24.89,"100,889,227,920.36",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
1889,BRA,Brazil,ZAF,South Africa,2018,"586,247,228.00","-598,004,391.90",NaN,"-5,991,082.19","6,168,495.60","1,728.00","196,975,010.60","59,084,600,353.00","691,297,141.80","105,049,913.80",0.00,11.00,11.00,23.00,"-598,004,391.90",0.00,20.19,7.46,19.10,24.80,20.35,18.47,0.00,0.11,0.15,0.20,0.22,0.12,0.17,0.28,"405,260,723,892.52","57,339,635.00",NaN,264.47,"5,484,091.39",5.58,26.73,17.86,"18,683,469,011.80",23.65,24.89,"100,889,227,920.36",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
2710,CHE,Switzerland,ZAF,South Africa,2018,"6,092,184,041.00","2,370,041,873.00",NaN,"178,998,693.00","-109,055,424.00","37,893.00","5,737,503,673.00","2,476,208,920.00","8,935,121,363.00","2,842,936,324.00",10.00,49.00,49.00,234.00,"2,370,041,873.00",21.59,22.53,10.54,22.47,21.63,22.91,21.77,2.40,0.11,0.15,0.20,0.22,0.12,0.17,0.28,"405,260,723,892.52","57,339,635.00",NaN,264.47,"120,259,649.95",5.58,26.73,17.86,"18,683,469,011.80",23.65,24.89,"100,889,227,920.36",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
3560,CHN,China (People’s Republic of),ZAF,South Africa,2018,"3,656,976,929.00","119,346,255.30",NaN,"63,004,666.45","83,823,687.07","4,283.00","1,446,680,487.00","858,030,430.70","4,729,600,500.00","1,071,850,205.00",3.00,61.00,61.00,110.00,"119,346,255.30",18.60,22.02,8.36,21.09,20.57,22.28,20.79,1.39,0.11,0.15,0.20,0.22,0.12,0.17,0.28,"405,260,723,892.52","57,339,635.00",NaN,264.47,"13,592,802.91",5.58,26.73,17.86,"18,683,469,011.80",23.65,24.89,"100,889,227,920.36",0.00,49.42,Africa,0.00,0.00,0.00,0.00,0.00,0.00
4130,CYM,Cayman Islands,ZAF,South Africa,2018,"42,421,601.48","-2,97

### Step 5.2. Perform the misalignment estimates, and all the remaining calculations needed

In [51]:
# Initialize a list to store the aggregate results
results_sample = []

# Start the estimates
misalignment_final_estimates_2018 = final_misalignment_2018[final_misalignment_2018['year'] == 2018].copy()
misalignment_final_estimates_2018 = calculate_misalignment(misalignment_final_estimates_2018, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Perform the groupby operation on 'iso_partner'
country_results_2018 = misalignment_final_estimates_2018.groupby(['iso_partner']).agg(
    negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
    positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
    theoretical_profit=('theoretical_profit', 'sum'),
    reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
).reset_index()

# Convert results to millions
country_results_2018['negative_misalignment'] = -country_results_2018['negative_misalignment'] / 1e6
country_results_2018['positive_misalignment'] = country_results_2018['positive_misalignment'] / 1e6
country_results_2018['theoretical_profit'] = country_results_2018['theoretical_profit'] / 1e6
country_results_2018['reported_profit'] = country_results_2018['reported_profit'] / 1e6

# Merge the unique columns back into the result
country_results_2018 = country_results_2018.merge(unique_columns, on='iso_partner', how='left')

# Calculate other relevant variables
country_results_2018['tax_revenue_loss'] = country_results_2018['negative_misalignment'] * country_results_2018['cit']
country_results_2018['tax_revenue_gain'] = country_results_2018['positive_misalignment'] * country_results_2018['etr_average_corrected']

country_results_2018['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
    country_results_2018['gvt_health_expenditure'] == 0, 
    np.nan, 
    country_results_2018['tax_revenue_loss'] / (country_results_2018['gvt_health_expenditure'] / 1e6)
)
    
country_results_2018['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
    country_results_2018['tax_revenue_current_usd'] == 0, 
    np.nan, 
    country_results_2018['tax_revenue_loss'] / (country_results_2018['tax_revenue_current_usd'] / 1e6)
)

# Calculate totals
total_positive_misalignment = country_results_2018['positive_misalignment'].sum()
total_negative_misalignment = country_results_2018['negative_misalignment'].sum()
total_profits = country_results_2018['reported_profit'].sum()
misaligned_of_total_profits = total_positive_misalignment / total_profits
total_tax_revenue_loss = country_results_2018['tax_revenue_loss'].sum()
total_tax_revenue_gain = country_results_2018['tax_revenue_gain'].sum()
average_tax_revenue_loss_pct_of_gvt_health_expenditure = country_results_2018['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
average_tax_revenue_loss_pct_of_total_tax_revenues = country_results_2018['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

print(f"Year {2018}: Positive Misalignment: {total_positive_misalignment}, Negative Misalignment: {total_negative_misalignment}, Shifted of total profits: {misaligned_of_total_profits}, "
        f"Total tax revenue loss: {total_tax_revenue_loss}, Total tax revenue gain: {total_tax_revenue_gain}")

# Calculate countries' fractions of totals
country_results_2018['tax_revenue_loss_caused_pct_of_total'] = country_results_2018['positive_misalignment'] / total_positive_misalignment
country_results_2018['tax_revenue_loss_caused_usd'] = country_results_2018['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss
country_results_2018['tax_revenue_loss_suffered_pct_of_total'] = country_results_2018['tax_revenue_loss'] / total_tax_revenue_loss

#country_results_2018 = country_results_2018[['iso_partner', 'partner_jurisdiction', 'negative_misalignment',
#   'tax_revenue_loss', 'tax_revenue_loss_suffered_pct_of_total', 'tax_revenue_loss_pct_of_gvt_health_expenditure',
#   'tax_revenue_loss_pct_of_total_tax_revenues', 'positive_misalignment', 'tax_revenue_gain', 
#   'tax_revenue_loss_caused_usd', 'tax_revenue_loss_caused_pct_of_total', 'etr_average_corrected', 'cit',
#   'region_tjn', 'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

country_results_2018 = country_results_2018.sort_values(by='iso_partner')
country_results_2018.to_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2018.csv', index=False) # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE
    
# Append aggregate results to the list
results_sample.append({
    'year': 2018,
    'total_positive_misalignment': total_positive_misalignment,
    'total_negative_misalignment': total_negative_misalignment,
    'total_profits': total_profits,
    'misaligned_of_total_profits': misaligned_of_total_profits,
    'total_tax_revenue_loss': total_tax_revenue_loss,
    'total_tax_revenue_gain': total_tax_revenue_gain,
    'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure,
    'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues
})

# Convert aggregate results to a DataFrame
results_sample_df = pd.DataFrame(results_sample)

# Save the aggregated results to a CSV or Excel file
results_sample_df.to_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_aggregate_results_2018.csv', index=False)  # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE


Year 2018: Positive Misalignment: 994585.760249356, Negative Misalignment: 994585.760249356, Shifted of total profits: 0.16605281429792812, Total tax revenue loss: 270125.77703681315, Total tax revenue gain: 86940.77247087924


C:\Users\aliso\AppData\Local\Temp\ipykernel_10016\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


In [52]:
# Ensure the inputs are numeric (optional but robust)
cols_num = ['misaligned_profit', 'profit_loss_before_income_tax_corrected']
misalignment_final_estimates_2018[cols_num] = misalignment_final_estimates_2018[cols_num].apply(
    pd.to_numeric, errors='coerce'
).fillna(0)

# By headquarter (reporting) country
hq_all = (
    misalignment_final_estimates_2018
    .groupby('iso_parent', as_index=False)
    .agg(
        shifted_out=('misaligned_profit', lambda s: (-s.clip(upper=0)).sum()),  # magnitude of negatives
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum'),
        shifted_in=('misaligned_profit', lambda s: s.clip(lower=0).sum())       # optional
    )
)

# Fractions (per HQ)
hq_all['fraction_shifted_out'] = np.where(
    hq_all['reported_profit'] == 0, np.nan, hq_all['shifted_out'] / hq_all['reported_profit']
)
hq_all['fraction_shifted_out_pct'] = 100 * hq_all['fraction_shifted_out']

# (optional)
hq_all['fraction_shifted_in'] = np.where(
    hq_all['reported_profit'] == 0, np.nan, hq_all['shifted_in'] / hq_all['reported_profit']
)
hq_all['fraction_shifted_in_pct'] = 100 * hq_all['fraction_shifted_in']

# Save
hq_all.sort_values('iso_parent').to_csv(
    f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_hq_fractions_2018_all.csv', index=False
)

### 5.3 US MNEs

In [53]:
import os
import numpy as np
import pandas as pd

# ================== Config (pick ONE year) ==================
YEAR = 2018  # <— change this per file (e.g., 2018, 2019, …)
OUTPUT_DIR = f"{output_tables}/Final_Full_CBCR_Datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ETR_MAX = 0.15
WEIGHTS = [0.5, 0, 0, 0.5, 0, 0, 0, 0]  # same as your example

# ================== Load the per-year df by name ==================
df_name = f"final_misalignment_{YEAR}"
if df_name not in globals():
    raise NameError(f"{df_name} not found in globals(). Make sure you created it earlier.")
final_misalignment_year = globals()[df_name]

# ================== Collect HQs (keeps your year filter) ==================
hq_list = (
    final_misalignment_year.loc[final_misalignment_year["year"] == YEAR, "iso_parent"]
    .dropna().astype(str).str.upper().unique()
)
hq_list = sorted(hq_list)
print(f"[{YEAR}] HQs: {', '.join(hq_list)}")

# ================== Run for this single year ==================
for HQ in hq_list:
    # 1) Initialize a list to store the aggregate results (per HQ & YEAR)
    results_sample = []

    # 2) Run the estimates — EXACT same filter structure
    mask = (final_misalignment_year['year'] == YEAR) & (final_misalignment_year['iso_parent'] == HQ)
    misalignment_final_estimates = final_misalignment_year.loc[mask].copy()

    if misalignment_final_estimates.empty:
        print(f"[{YEAR}][{HQ}] No rows; skipping.")
        continue

    misalignment_final_estimates = calculate_misalignment(
        misalignment_final_estimates,
        etr_max=ETR_MAX,
        weights=WEIGHTS
    )

    # 3) Groupby iso_partner
    country_results = misalignment_final_estimates.groupby(['iso_partner']).agg(
        negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
        theoretical_profit=('theoretical_profit', 'sum'),
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
    ).reset_index()

    # 4) Convert to millions
    country_results['negative_misalignment'] = -country_results['negative_misalignment'] / 1e6
    country_results['positive_misalignment'] =  country_results['positive_misalignment'] / 1e6
    country_results['theoretical_profit']   =  country_results['theoretical_profit'] / 1e6
    country_results['reported_profit']      =  country_results['reported_profit'] / 1e6

    # 5) Merge the unique columns
    country_results = country_results.merge(unique_columns, on='iso_partner', how='left')

    # 6) Other variables (same logic as yours)
    country_results['tax_revenue_loss'] = country_results['negative_misalignment'] * country_results['cit']
    country_results['tax_revenue_gain'] = country_results['positive_misalignment'] * country_results['etr_average_corrected']

    country_results['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
        country_results['gvt_health_expenditure'] == 0, 
        np.nan, 
        country_results['tax_revenue_loss'] / (country_results['gvt_health_expenditure'] / 1e6)
    )
    country_results['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
        country_results['tax_revenue_current_usd'] == 0, 
        np.nan, 
        country_results['tax_revenue_loss'] / (country_results['tax_revenue_current_usd'] / 1e6)
    )

    # 7) Totals
    total_positive_misalignment = country_results['positive_misalignment'].sum()
    total_negative_misalignment = country_results['negative_misalignment'].sum()
    total_profits               = country_results['reported_profit'].sum()
    misaligned_of_total_profits = total_positive_misalignment / total_profits if total_profits != 0 else np.nan
    total_tax_revenue_loss      = country_results['tax_revenue_loss'].sum()
    total_tax_revenue_gain      = country_results['tax_revenue_gain'].sum()
    average_loss_pct_health     = country_results['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
    average_loss_pct_taxrev     = country_results['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

    print(
        f"[{YEAR}][{HQ}] +mis={total_positive_misalignment:.3f}m, "
        f"-mis={total_negative_misalignment:.3f}m, "
        f"share={misaligned_of_total_profits if pd.notna(misaligned_of_total_profits) else np.nan:.3f}, "
        f"loss={total_tax_revenue_loss:.3f}m, gain={total_tax_revenue_gain:.3f}m"
    )

    # 9) Fractions of totals
    country_results['tax_revenue_loss_caused_pct_of_total'] = (
        country_results['positive_misalignment'] / total_positive_misalignment
        if total_positive_misalignment != 0 else np.nan
    )
    country_results['tax_revenue_loss_caused_usd'] = (
        country_results['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss
    )
    country_results['tax_revenue_loss_suffered_pct_of_total'] = (
        country_results['tax_revenue_loss'] / total_tax_revenue_loss
        if total_tax_revenue_loss != 0 else np.nan
    )

    # Save per-HQ country file
    country_results = country_results.sort_values(by='iso_partner')
    per_hq_file = f'{OUTPUT_DIR}/SOTJ_sample_countries_{YEAR}_{HQ}MNEs.csv'
    country_results.to_csv(per_hq_file, index=False)

    # 10) Append aggregate results to the list (per HQ & year)
    results_sample = [{
        'year': YEAR,
        'total_positive_misalignment': total_positive_misalignment,
        'total_negative_misalignment': total_negative_misalignment,
        'total_profits': total_profits,
        'misaligned_of_total_profits': misaligned_of_total_profits,
        'total_tax_revenue_loss': total_tax_revenue_loss,
        'total_tax_revenue_gain': total_tax_revenue_gain,
        'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_loss_pct_health,
        'average_tax_revenue_loss_pct_of_total_tax_revenues': average_loss_pct_taxrev
    }]

    # 11) Save the aggregated results (per HQ & year)
    results_sample_df = pd.DataFrame(results_sample)
    agg_file = f'{OUTPUT_DIR}/SOTJ_sample_aggregate_results_{YEAR}_{HQ}MNEs.csv'
    results_sample_df.to_csv(agg_file, index=False)


[2018] HQs: ARG, AUS, AUT, BEL, BMU, BRA, CAN, CHE, CHL, CHN, CYM, CZE, DEU, DNK, ESP, FIN, FRA, GBR, GRC, HKG, HUN, IDN, IMN, IND, IRL, ITA, JPN, KOR, LTU, LUX, LVA, MEX, MYS, NLD, NOR, NZL, PAN, PER, POL, ROU, SAU, SGP, SVN, SWE, USA, ZAF


[2018][ARG] +mis=535.017m, -mis=535.017m, share=0.048, loss=149.717m, gain=51.731m
[2018][AUS] +mis=9032.782m, -mis=9032.782m, share=0.099, loss=2478.279m, gain=754.095m


C:\Users\aliso\AppData\Local\Temp\ipykernel_10016\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_10016\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=Fals

[2018][AUT] +mis=13793.076m, -mis=13793.076m, share=0.313, loss=3919.650m, gain=1963.865m
[2018][BEL] +mis=109073.299m, -mis=109073.299m, share=0.774, loss=32980.435m, gain=9115.143m
[2018][BMU] +mis=6860.637m, -mis=6860.637m, share=0.335, loss=1757.661m, gain=251.146m


C:\Users\aliso\AppData\Local\Temp\ipykernel_10016\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_10016\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=Fals

...

[2018][ROU] +mis=1871.226m, -mis=1871.226m, share=0.095, loss=360.512m, gain=141.049m
[2018][SAU] +mis=90.660m, -mis=90.660m, share=0.000, loss=15.346m, gain=2.520m
[2018][SGP] +mis=11538.391m, -mis=11538.391m, share=0.309, loss=3272.696m, gain=744.752m


C:\Users\aliso\AppData\Local\Temp\ipykernel_10016\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_10016\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=Fals

[2018][SVN] +mis=39.716m, -mis=39.716m, share=0.074, loss=5.747m, gain=5.242m
[2018][SWE] +mis=48540.516m, -mis=48540.516m, share=0.313, loss=13794.009m, gain=6911.224m


C:\Users\aliso\AppData\Local\Temp\ipykernel_10016\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_10016\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=Fals

[2018][USA] +mis=202322.275m, -mis=202322.275m, share=0.256, loss=55667.551m, gain=10132.284m
[2018][ZAF] +mis=25919.786m, -mis=25919.786m, share=0.408, loss=7323.059m, gain=2274.673m


## Step 6. Checking the datasets

### Step 6.1. Checking the countries


In [54]:
# Open the CSV file f'{output_tables}/Scaling_Mario/SOTJ_sample_2016.csv'
sotj_2018_countries = pd.read_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2018.csv')
sotj_2018_countries

,iso_partner,negative_misalignment,positive_misalignment,theoretical_profit,reported_profit,partner_jurisdiction,etr_average_corrected,cit,tax_revenue_current_usd,gvt_health_expenditure,region_tjn,ukt,oecd,oecd_oct,nld_oct,tax_revenue_loss,tax_revenue_gain,tax_revenue_loss_pct_of_gvt_health_expenditure,tax_revenue_loss_pct_of_total_tax_revenues,tax_revenue_loss_caused_pct_of_total,tax_revenue_loss_caused_usd,tax_revenue_loss_suffered_pct_of_total
0,ABW,3.69,0.27,72.93,101.65,Aruba,0.15,0.25,NaN,NaN,Caribbean/American isl.,0.00,0.00,1.00,1.00,0.92,0.04,NaN,NaN,0.00,0.07,0.00
1,AFG,-0.00,6.95,2.44,9.38,Afghanistan,0.14,0.20,NaN,"101,744,864.14",Asia,0.00,0.00,0.00,0.00,-0.00,1.00,-0.00,NaN,0.00,1.89,-0.00
2,AGO,73.92,653.89,386.53,"4,599.97",Angola,0.41,0.30,"7,662,842,901.72","1,118,847,264.51",Africa,0.00,0.00,0.00,0.00,22.18,266.25,0.02,0.00,0.00,177.59,0.00
3,AIA,-0.00,40.64,1.46,42.10,Anguilla,0.00,0.00,NaN,NaN,Caribbean/American isl.,1.00,0.00,1.00,0.00,-0.00,0.00,NaN,NaN,0.00,11.04,-0.00
4,ALB,97.83,4.66,80.79,-60.53,Albania,0.08,0.15,"2,810,905,467.47","435,678,213.19",Europe,0.00,0.00,0.00,0.00,14.67,0.36,0.03,0.01,0.00,1.27,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208,XKV,1.63,0.00,4.41,1.91,Kosovo,0.02,0.10,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.16,0.00,NaN,NaN,0.00,0.00,0.00
209,YEM,0.48,0.52,1.51,3.98,Yemen,0.45,0.20,NaN,NaN,Asia,0.00,0.00,0.00,0.00,0.10,0.23,NaN,NaN,0.00,0.14,0.00
210,ZAF,"13,955.71",0.00,"46,841.07","29,748.26",South Africa,0.17,0.28,"100,889,227,920.36","18,683,469,011.80",Africa,0.00,0.00,0.00,0.00,"3,907.60",0.00,0.21,0.04,0.00,0.00,0.01
211,ZMB,"1,740.04",0.00,"1,460.55","1,089.09",Zambia,0.15,0.35,"4,364,306,937.63","549,388,229.53",Africa,0.00,0.00,0.00,0.00,609.02,0.00,1.11,0.14,0.00,0.00,0.00


### Step 6.2. Checking the aggregate results

In [55]:
sotj_2018_aggregate = pd.read_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_aggregate_results_2018.csv')
sotj_2018_aggregate

,year,total_positive_misalignment,total_negative_misalignment,total_profits,misaligned_of_total_profits,total_tax_revenue_loss,total_tax_revenue_gain,average_tax_revenue_loss_pct_of_gvt_health_expenditure,average_tax_revenue_loss_pct_of_total_tax_revenues
0,2018,"994,585.76","994,585.76","5,989,574.85",0.17,"270,125.78","86,940.77",0.24,0.03
